# De una web a una tabla de hallazgos

**Taller IA para autoridades de consumo.** Trabajamos sobre *ElectroEjemplo*, una tienda **ficticia** creada para esta formación. Ningún producto, precio ni empresa es real.

Qué vamos a hacer, paso a paso:

1. Comprobar si el sitio permite visitar cada página (`robots.txt`)
2. Descargar el HTML de la web
3. Convertir los productos en una tabla
4. Marcar posibles incumplimientos
5. Guardar los hallazgos en un Excel

Todo el código lo ha escrito una IA. Vuestro trabajo es **entenderlo, ejecutarlo y verificarlo**.

In [ ]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
from difflib import SequenceMatcher
from urllib import robotparser
from urllib.parse import urljoin

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

## Paso 0. Configuración

Pega aquí la dirección de la tienda ficticia que os damos en clase.
Si no hay conexión, sube la carpeta `tienda` a Colab (icono de carpeta de la izquierda) y deja `BASE_URL = ""`.

In [ ]:
BASE_URL = "https://globalcobots-bot.github.io/electroejemplo/"
CARPETA_LOCAL = "tienda"
USER_AGENT = "TallerConsumoBot/1.0 (formacion)"

def descargar(ruta):
    """Devuelve el HTML de una ruta de la tienda."""
    if BASE_URL:
        r = requests.get(urljoin(BASE_URL, ruta), headers={"User-Agent": USER_AGENT}, timeout=15)
        r.raise_for_status()
        return r.text
    with open(f"{CARPETA_LOCAL}/{ruta}", encoding="utf-8") as f:
        return f.read()

## Paso 1. Antes de pedir nada: ¿qué dice el robots.txt?

El archivo `robots.txt` indica qué zonas del sitio no quieren que visiten programas automáticos.
Es una norma de cortesía técnica y un indicio de la voluntad del titular, así que **se comprueba siempre antes de recopilar**.

In [ ]:
print(descargar("robots.txt"))

In [ ]:
robots = robotparser.RobotFileParser()
robots.parse(descargar("robots.txt").splitlines())

def puede_visitar(ruta):
    return robots.can_fetch(USER_AGENT, "/" + ruta)

paginas = ["index.html", "condiciones.html", "checkout.html",
           "producto-altavoz.html", "privado/clientes.html"]

for ruta in paginas:
    estado = "permitido" if puede_visitar(ruta) else "VETADO: no se descarga"
    print(f"{ruta:26} -> {estado}")

## Paso 2. Descargar el HTML y mirar su estructura

Es lo mismo que hace el navegador al pulsar **Inspeccionar**. Solo que aquí lo hace Python.

In [ ]:
html = descargar("index.html")
print(len(html), "caracteres de HTML")
print(html[:700])

In [ ]:
soup = BeautifulSoup(html, "html.parser")
print("Título de la página:", soup.title.get_text())
print("Productos encontrados:", len(soup.select("article.producto")))

## Paso 3. Los productos, en una tabla

`BeautifulSoup` recorre el árbol HTML. Cada producto está dentro de un `<article class="producto">`.

In [ ]:
def euros(texto):
    """Convierte '199,00 €' en 199.0"""
    if not texto:
        return None
    return float(texto.replace("€", "").replace(".", "").replace(",", ".").strip())

filas = []
for art in soup.select("article.producto"):
    tachado = art.select_one(".precio-antes")
    ahora = art.select_one(".precio-ahora")
    urgencia = art.select_one(".urgencia")
    filas.append({
        "sku": art["data-sku"],
        "producto": art.select_one(".nombre").get_text(strip=True),
        "precio_ahora": euros(ahora.get_text()) if ahora else None,
        "precio_tachado": euros(tachado.get_text()) if tachado else None,
        "precio_previo_30d": float(art["data-precio-previo-30d"]),
        "urgencia": urgencia.get_text(strip=True) if urgencia else "",
        "notas": " | ".join(n.get_text(strip=True) for n in art.select(".nota")),
    })

productos = pd.DataFrame(filas)
productos

## Paso 4. Primer hallazgo: rebajas sospechosas

Regla que vamos a comprobar: el precio anterior mostrado debe coincidir con el **menor precio aplicado en los 30 días previos**.
En esta tienda de práctica, esa información viene en el atributo `data-precio-previo-30d`, que simula un histórico de precios.

In [ ]:
productos["rebaja_dudosa"] = productos.apply(
    lambda f: pd.notna(f.precio_tachado) and abs(f.precio_tachado - f.precio_previo_30d) > 0.01,
    axis=1,
)
productos["urgencia_repetida"] = productos.urgencia.duplicated(keep=False) & (productos.urgencia != "")

hallazgos_precios = productos[productos.rebaja_dudosa | productos.urgencia_repetida]
hallazgos_precios[["sku", "producto", "precio_tachado", "precio_previo_30d", "precio_ahora", "rebaja_dudosa", "urgencia_repetida"]]

**Pregunta para el grupo:** ¿qué producto tiene una rebaja correcta y por qué no aparece marcado?

## Paso 5. Las condiciones de venta

Aquí las palabras clave sirven para localizar, pero no para decidir. Una IA clasifica mejor un texto jurídico.
Eso lo haremos en la segunda parte del taller.

In [ ]:
cond = BeautifulSoup(descargar("condiciones.html"), "html.parser")

clausulas = []
for h2 in cond.select("main h2"):
    parrafo = h2.find_next_sibling("p")
    clausulas.append({"clausula": h2.get_text(strip=True), "texto": parrafo.get_text(strip=True)})
clausulas = pd.DataFrame(clausulas)

senales = ["sin previo aviso", "renuncia", "en ningún caso", "bajo ningún concepto",
           "6 meses", "48 horas", "cedidos a terceras", "sin comunicarlo"]

clausulas["senales"] = clausulas.texto.apply(
    lambda t: ", ".join(s for s in senales if s in t.lower())
)
clausulas[clausulas.senales != ""][["clausula", "senales"]]

## Paso 6. El carrito: casillas ya marcadas

Un formulario HTML marca una casilla de serie con el atributo `checked`. Eso se ve en el código aunque la pantalla parezca limpia.

In [ ]:
def casillas_de(html_pagina, origen):
    s = BeautifulSoup(html_pagina, "html.parser")
    return [{
        "pagina": origen,
        "campo": i.get("name"),
        "texto": i.parent.get_text(" ", strip=True),
        "premarcada": i.has_attr("checked"),
    } for i in s.select("input[type=checkbox]")]

casillas = pd.DataFrame(casillas_de(html, "index.html") + casillas_de(descargar("checkout.html"), "checkout.html"))
casillas[casillas.premarcada]

## Paso 7. Reseñas con aspecto de copia

Buscamos reseñas del mismo día con un texto casi idéntico.

In [ ]:
rv = BeautifulSoup(descargar("producto-altavoz.html"), "html.parser")

resenas = []
for r in rv.select(".resena"):
    partes = r.get_text("\n", strip=True).split("\n")
    resenas.append({
        "autor": partes[0],
        "fecha": r["data-fecha"],
        "estrellas": int(re.search(r"(\d) estrellas", r.get_text()).group(1)),
        "texto": partes[-1],
    })
resenas = pd.DataFrame(resenas)

sospechosas = set()
for i in range(len(resenas)):
    for j in range(i + 1, len(resenas)):
        misma_fecha = resenas.fecha[i] == resenas.fecha[j]
        parecido = SequenceMatcher(None, resenas.texto[i], resenas.texto[j]).ratio()
        if misma_fecha and parecido > 0.9:
            sospechosas.update([i, j])

resenas["sospechosa"] = resenas.index.isin(sospechosas)
resenas

## Paso 8. Guardar los hallazgos

Un Excel con una hoja por tipo de hallazgo. Es la base para el informe del inspector.
**Todo hallazgo se verifica a mano en la web antes de usarlo como evidencia.**

In [ ]:
with pd.ExcelWriter("hallazgos_electroejemplo.xlsx") as w:
    hallazgos_precios.to_excel(w, sheet_name="precios_y_urgencia", index=False)
    clausulas[clausulas.senales != ""].to_excel(w, sheet_name="clausulas", index=False)
    casillas[casillas.premarcada].to_excel(w, sheet_name="casillas_premarcadas", index=False)
    resenas[resenas.sospechosa].to_excel(w, sheet_name="resenas_sospechosas", index=False)

print("Guardado: hallazgos_electroejemplo.xlsx")

In [ ]:
try:
    from google.colab import files
    files.download("hallazgos_electroejemplo.xlsx")
except ImportError:
    print("Fuera de Colab: el archivo está en la carpeta de trabajo.")

## Ejercicio con IA

Copia este código en ChatGPT o Claude y pide cambios. Ejemplos de peticiones:

1. "Añade una columna con el porcentaje real de descuento sobre el precio previo de 30 días."
2. "Explícame línea a línea el bucle del Paso 3 como si no supiera programar."
3. "Crea una función que haga lo mismo para otra página de producto con la misma estructura."

**Comprueba siempre lo que te devuelve la IA:** ejecuta el código, compara la tabla con la web y revisa que no se haya inventado ningún campo.

## Límites que conviene recordar

- Comprobar `robots.txt` y los términos de uso antes de recopilar
- Pedir pocas páginas y con pausas, sin saturar el servidor
- No guardar datos personales que no hagan falta, por ejemplo nombres en reseñas
- Lo recopilado es un indicio. La evidencia se confirma y se documenta con captura y fecha